# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rohith84/Flyrank-Week-1-/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice

This project uses a weighted scoring and ranking approach for the Refresh / Content Opportunity Scoring lane.

The W04 baseline uses a simple rule based mainly on search visibility and CTR. For W05, I extend this baseline into a multi-signal Opportunity Score using observed impressions, CTR, average search position, and engagement.

A scoring approach fits this lane because the goal is to prioritize pages for review rather than automatically predict a future Google ranking. The score produces a ranked queue that can be compared against the observed outcome using a defined ranking metric.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split Design

The evaluation uses a client-grouped holdout so that pages belonging to the same client are not intentionally mixed across the evaluation groups.

The baseline and Opportunity Score are evaluated on the same held-out pages and using the same ranking metric.

The features used to construct the scores do not include label-derived fields such as `trend_direction` or `trend_pct`.

This design provides a more honest comparison than evaluating both methods on different subsets of data.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

sample = pd.read_parquet(file_path)

print("Rows loaded:", len(sample))
print("Columns:", len(sample.columns))

Rows loaded: 11694072
Columns: 31


In [3]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

df = pd.read_parquet(file_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumns:")
print(df.columns.tolist())

Rows: 11694072
Columns: 31

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [4]:
print("Number of columns:", len(df.columns))
print("\nColumns in your W05 dataset:")
for col in df.columns:
    print(col)

Number of columns: 31

Columns in your W05 dataset:
report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


In [5]:
print(df.head(2).T)

                                                 0                         1
report_date                             2026-06-01                2026-06-01
client_hash_id             client_3ffa76342f366962   client_3ffa76342f366962
content_hash_id           content_1a6296faee432dae  content_73f21e612565035a
client_has_gsc                                True                      True
client_has_ga4                                True                      True
gsc_data_available                           False                     False
ga4_data_available                           False                     False
gsc_impressions                                  0                         0
gsc_clicks                                       0                         0
gsc_sum_position                               0.0                       0.0
gsc_avg_position                               NaN                       NaN
ga4_pageviews                                  0.0                       0.0

In [6]:
print("Possible date/time columns:")
for col in df.columns:
    if any(x in col.lower() for x in ["date", "month", "day", "time"]):
        print(col)

print("\nPage/content identifier columns:")
for col in df.columns:
    if any(x in col.lower() for x in ["content", "page", "url", "hash"]):
        print(col)

print("\nClient/group columns:")
for col in df.columns:
    if "client" in col.lower():
        print(col)

Possible date/time columns:
report_date
month

Page/content identifier columns:
client_hash_id
content_hash_id
ga4_pageviews

Client/group columns:
client_hash_id
client_has_gsc
client_has_ga4


In [7]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [8]:
print("Report date range:")
print(df["report_date"].min(), "to", df["report_date"].max())

print("\nUnique months:")
print(sorted(df["month"].dropna().astype(str).unique()))

print("\nNumber of unique months:",
      df["month"].nunique())

print("\nRows per month:")
print(
    df.groupby("month")
      .size()
      .sort_index()
)

Report date range:
2026-06-01 to 2026-06-30

Unique months:
['2026-06']

Number of unique months: 1

Rows per month:
month
2026-06    11694072
dtype: int64


In [9]:
import numpy as np
import pandas as pd

# Get unique clients only
clients = df["client_hash_id"].drop_duplicates().reset_index(drop=True)

# Deterministic client-level split
rng = np.random.default_rng(42)

shuffled_clients = clients.to_numpy().copy()
rng.shuffle(shuffled_clients)

split_point = int(len(shuffled_clients) * 0.80)

train_clients = set(shuffled_clients[:split_point])
test_clients = set(shuffled_clients[split_point:])

print("Total clients:", len(clients))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))

print(
    "Client overlap:",
    len(train_clients & test_clients)
)

Total clients: 65
Train clients: 52
Test clients: 13
Client overlap: 0


In [10]:
train_mask = df["client_hash_id"].isin(train_clients)
test_mask = df["client_hash_id"].isin(test_clients)

print("Train rows:", train_mask.sum())
print("Test rows:", test_mask.sum())

Train rows: 8883542
Test rows: 2810530


In [11]:
page_df = (
    df.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg({
        "gsc_impressions": "sum",
        "gsc_clicks": "sum",
        "gsc_avg_position": "mean",
        "ga4_pageviews": "sum",
        "ga4_sessions": "sum",
        "ga4_engaged_sessions": "sum",
        "ga4_total_engagement_sec": "sum",
        "sessions_organic": "sum",
        "sessions_ai": "sum"
    })
)

print("Original rows:", len(df))
print("Page-level rows:", len(page_df))
print("Unique clients:", page_df["client_hash_id"].nunique())
print("Unique pages:", page_df["content_hash_id"].nunique())

Original rows: 11694072
Page-level rows: 409205
Unique clients: 65
Unique pages: 409205


In [12]:
page_df["ctr"] = (
    page_df["gsc_clicks"] /
    page_df["gsc_impressions"].replace(0, np.nan)
).fillna(0)

page_df["engagement_per_session"] = (
    page_df["ga4_total_engagement_sec"] /
    page_df["ga4_sessions"].replace(0, np.nan)
).fillna(0)

print(page_df[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "engagement_per_session"
]].describe())

       gsc_impressions     gsc_clicks  gsc_avg_position            ctr  \
count    409205.000000  409205.000000     208636.000000  409205.000000   
mean        528.329009       2.954795         22.929469       0.003590   
std        3698.473081     326.984994         22.844836       0.023825   
min           0.000000       0.000000          0.000000       0.000000   
25%           0.000000       0.000000          6.936390       0.000000   
50%           1.000000       0.000000         12.750000       0.000000   
75%          98.000000       0.000000         32.250000       0.000000   
max      615012.000000  152170.000000        579.000000       1.000000   

       engagement_per_session  
count            409205.00000  
mean                  1.76877  
std                  18.04708  
min                   0.00000  
25%                   0.00000  
50%                   0.00000  
75%                   0.00000  
max                3235.00000  


In [13]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        page_df,
        groups=page_df["client_hash_id"]
    )
)

train_df = page_df.iloc[train_idx].copy()
test_df = page_df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

overlap = (
    set(train_df["client_hash_id"])
    &
    set(test_df["client_hash_id"])
)

print("Client overlap:", len(overlap))

Train rows: 354310
Test rows: 54895
Train clients: 52
Test clients: 13
Client overlap: 0


In [14]:
high_impression_threshold = (
    train_df.loc[
        train_df["gsc_impressions"] > 0,
        "gsc_impressions"
    ].quantile(0.75)
)

low_ctr_threshold = (
    train_df.loc[
        train_df["gsc_impressions"] > 0,
        "ctr"
    ].quantile(0.25)
)

print(
    "High impression threshold:",
    round(high_impression_threshold, 4)
)

print(
    "Low CTR threshold:",
    round(low_ctr_threshold, 4)
)

High impression threshold: 524.0
Low CTR threshold: 0.0


In [25]:
print(
    "Pages with zero CTR in training:",
    (train_df["ctr"] == 0).sum()
)

print(
    "Percentage with zero CTR:",
    round(
        (train_df["ctr"] == 0).mean() * 100,
        2
    ),
    "%"
)

Pages with zero CTR in training: 284013
Percentage with zero CTR: 80.16 %


In [15]:
test_df["baseline_score"] = (
    (
        test_df["gsc_impressions"]
        >= high_impression_threshold
    ).astype(int)
    +
    (
        (test_df["gsc_impressions"] > 0)
        &
        (test_df["ctr"] <= low_ctr_threshold)
    ).astype(int)
)

In [16]:
def scale_with_train_range(series, minimum, maximum):
    if maximum == minimum:
        return pd.Series(0.0, index=series.index)

    return (
        (series - minimum) /
        (maximum - minimum)
    ).clip(0, 1)

In [17]:
imp_min = train_df["gsc_impressions"].min()
imp_max = train_df["gsc_impressions"].max()

ctr_min = train_df["ctr"].min()
ctr_max = train_df["ctr"].max()

pos_min = train_df["gsc_avg_position"].min()
pos_max = train_df["gsc_avg_position"].max()

eng_min = train_df["engagement_per_session"].min()
eng_max = train_df["engagement_per_session"].max()

In [18]:
test_df["visibility_component"] = scale_with_train_range(
    test_df["gsc_impressions"],
    imp_min,
    imp_max
)

test_df["ctr_component"] = 1 - scale_with_train_range(
    test_df["ctr"],
    ctr_min,
    ctr_max
)

test_df["position_component"] = scale_with_train_range(
    test_df["gsc_avg_position"],
    pos_min,
    pos_max
)

test_df["engagement_component"] = 1 - scale_with_train_range(
    test_df["engagement_per_session"],
    eng_min,
    eng_max
)

In [19]:
test_df["opportunity_score"] = (
    0.35 * test_df["visibility_component"]
    + 0.30 * test_df["ctr_component"]
    + 0.20 * test_df["position_component"]
    + 0.15 * test_df["engagement_component"]
)

In [20]:
baseline_top50 = (
    test_df
    .sort_values(
        "baseline_score",
        ascending=False
    )
    .head(50)
)

opportunity_top50 = (
    test_df
    .sort_values(
        "opportunity_score",
        ascending=False
    )
    .head(50)
)

print("Baseline top-50:", len(baseline_top50))
print("Opportunity Score top-50:", len(opportunity_top50))

Baseline top-50: 50
Opportunity Score top-50: 50


In [21]:
baseline_ids = set(
    baseline_top50["content_hash_id"]
)

opportunity_ids = set(
    opportunity_top50["content_hash_id"]
)

overlap_count = len(
    baseline_ids & opportunity_ids
)

print("Top-50 overlap:", overlap_count)
print(
    "Unique pages surfaced by Opportunity Score:",
    len(opportunity_ids - baseline_ids)
)

Top-50 overlap: 0
Unique pages surfaced by Opportunity Score: 50


In [22]:
display(
    opportunity_top50[[
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "engagement_per_session",
        "opportunity_score"
    ]].head(20)
)

,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,engagement_per_session,opportunity_score
385422,content_ef7013c86d07aa99,150344,692,0.004603,6.814726,2.000000,0.536440
301289,content_0697f8f82ae8629b,1,0,0.000000,212.000000,0.000000,0.523230
301705,content_2f6a99be9fc67cdc,1,0,0.000000,209.000000,0.000000,0.522194
33827,content_2b3530da58784950,32,0,0.000000,158.175000,0.000000,0.504656
34511,content_f1f15b68bc8a6131,21,0,0.000000,148.452778,1.500000,0.501221
34533,content_fa0f51490fa5c25e,22,0,0.000000,146.541667,0.000000,0.500631
34094,content_7efd3b8b440e01fe,28,0,0.000000,138.853274,0.000000,0.497979
34289,content_ba4d8bdd345338d6,36,0,0.000000,135.057292,0.000000,0.496672
33814,content_27eccaa7e7a4481e,37,0,0.000000,133.090986,0.000000,0.495994
389065,content_8ffabfd27d2295b2,1,0,0.000000,133.000000,0.000000,0.495942


## Baseline Comparison

The W04 baseline is a transparent rule-based score using two signals:

- High impressions
- Low CTR

The proposed W05 Opportunity Score uses four signals:

- Impressions
- CTR
- Average position
- Engagement per session

Both methods are applied to the same client-grouped evaluation population.

Because the available warehouse release contains only June 2026, there is no future observation window available to measure whether the ranked pages subsequently declined. Therefore, this comparison evaluates the ranking composition and overlap rather than claiming predictive superiority.

The Opportunity Score is intended as directional decision support for prioritizing pages for human review.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [23]:
# Compare the pages selected by the two ranking approaches

baseline_top20 = (
    test_df
    .sort_values(
        ["baseline_score", "gsc_impressions"],
        ascending=[False, False]
    )
    .head(20)
)

opportunity_top20 = (
    test_df
    .sort_values(
        "opportunity_score",
        ascending=False
    )
    .head(20)
)

baseline_ids = set(baseline_top20["content_hash_id"])
opportunity_ids = set(opportunity_top20["content_hash_id"])

common_pages = baseline_ids & opportunity_ids
opportunity_only = opportunity_ids - baseline_ids
baseline_only = baseline_ids - opportunity_ids

print("Baseline top-20:", len(baseline_ids))
print("Opportunity Score top-20:", len(opportunity_ids))
print("Pages appearing in both:", len(common_pages))
print("Pages unique to Opportunity Score:", len(opportunity_only))
print("Pages unique to baseline:", len(baseline_only))

Baseline top-20: 20
Opportunity Score top-20: 20
Pages appearing in both: 0
Pages unique to Opportunity Score: 20
Pages unique to baseline: 20


In [24]:
# Inspect the pages uniquely surfaced by the Opportunity Score

opportunity_only_df = (
    opportunity_top20[
        opportunity_top20["content_hash_id"].isin(opportunity_only)
    ][[
        "content_hash_id",
        "gsc_impressions",
        "ctr",
        "gsc_avg_position",
        "engagement_per_session",
        "opportunity_score"
    ]]
)

display(opportunity_only_df)

,content_hash_id,gsc_impressions,ctr,gsc_avg_position,engagement_per_session,opportunity_score
385422,content_ef7013c86d07aa99,150344,0.004603,6.814726,2.000000,0.536440
301289,content_0697f8f82ae8629b,1,0.000000,212.000000,0.000000,0.523230
301705,content_2f6a99be9fc67cdc,1,0.000000,209.000000,0.000000,0.522194
33827,content_2b3530da58784950,32,0.000000,158.175000,0.000000,0.504656
34511,content_f1f15b68bc8a6131,21,0.000000,148.452778,1.500000,0.501221
34533,content_fa0f51490fa5c25e,22,0.000000,146.541667,0.000000,0.500631
34094,content_7efd3b8b440e01fe,28,0.000000,138.853274,0.000000,0.497979
34289,content_ba4d8bdd345338d6,36,0.000000,135.057292,0.000000,0.496672
33814,content_27eccaa7e7a4481e,37,0.000000,133.090986,0.000000,0.495994
389065,content_8ffabfd27d2295b2,1,0.000000,133.000000,0.000000,0.495942


## Errors and Interpretation

The Opportunity Score is not a predictive model, so errors are interpreted as potentially weak or questionable ranking recommendations rather than classification errors.

The score gives higher priority to pages that combine meaningful visibility with weaker CTR, weaker search position, or lower engagement.

Some pages may receive a high score because one signal is unusually strong while the other signals are weaker. These pages should therefore be reviewed by an editor before action is taken.

The client-grouped split prevents pages from the same client appearing in both the training-derived threshold calculation and evaluation population.

No future-window outcome or label-derived field was used because the available warehouse release contains only June 2026.

The analysis is therefore directional and intended for decision support, not proof that a recommended refresh will improve future search performance.

## Conclusion

The Opportunity Score produced a substantially different ranking from the W04 baseline, with zero overlap among the top 50 pages in this evaluation sample.

This indicates that adding CTR, average position, and engagement changes which pages are prioritized compared with the simpler baseline.

However, because the available warehouse release contains only June 2026, there is no future outcome available to determine which ranking is more accurate.

Therefore, this result should be interpreted as an observed difference in ranking behavior, not evidence that the Opportunity Score predicts future content performance better.

The score is intended as directional decision support and should be combined with human review before content changes are made.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.